# Baseline Models: Random Forest and XGBoost for Resale Flat Prices

## Imports and Set Ups

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import mysql.connector
from sqlalchemy import create_engine, text

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

In [41]:
# Load transform_resale_flat_prices
HDB_db = mysql.connector.connect(
	host='localhost',
	user='bt4301-project',
	passwd='password',
	database='HDB_Data'
)

str_sql = '''
SELECT 
    # resale flat attributes
    flat_type, flat_model, floor_area_sqm, resale_price, price_per_sqm,
    max_floor_lvl, total_dwelling_units, storey_mid, storey_ratio,
    remaining_lease_years, log_resale_price, building_age
    # locational attributes
    town, has_market_hawker, has_multistorey_carpark, dist_to_nearest_mrt_m,
    n_mrt_within_1km, dist_to_mall_m, dist_to_school_m, n_school_within_1km,
    dist_to_food_m, n_food_within_1km, dist_to_park_m, n_park_within_1km,
    dist_to_supermarket_m, n_supermarket_within_1km, dist_to_nearest_bus_stop_m,
    n_bus_stop_within_1km, dist_to_nearest_tourist_attraction_m, 
    dist_to_nearest_carpark_m, n_carparks_within_500m,
    # temporal attributes
    month_index, year, month
FROM transform_resale_flat_price;
'''

resale_df = pd.read_sql(sql=str_sql, con=HDB_db)

# View resale_df
print(f"Shape of resale_df: {resale_df.shape}")
resale_df.head()

/tmp/ipykernel_95718/185793924.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  resale_df = pd.read_sql(sql=str_sql, con=HDB_db)


Shape of resale_df: (222392, 33)


,flat_type,flat_model,floor_area_sqm,resale_price,price_per_sqm,max_floor_lvl,total_dwelling_units,storey_mid,storey_ratio,remaining_lease_years,log_resale_price,town,has_market_hawker,has_multistorey_carpark,dist_to_nearest_mrt_m,n_mrt_within_1km,dist_to_mall_m,dist_to_school_m,n_school_within_1km,dist_to_food_m,n_food_within_1km,dist_to_park_m,n_park_within_1km,dist_to_supermarket_m,n_supermarket_within_1km,dist_to_nearest_bus_stop_m,n_bus_stop_within_1km,dist_to_nearest_tourist_attraction_m,dist_to_nearest_carpark_m,n_carparks_within_500m,month_index,year,month
0,2 ROOM,Improved,44.0000,"233,000.0000","5,295.4500",12,154,5.0000,0.4167,59.3300,12.3588,41,0,0,275.2900,1,182.3600,433.4460,4,120.0910,35,386.2050,3,205.0910,6,145.9130,53,"3,142.4800",41.1421,21,1,2017,2
1,2 ROOM,Improved,44.0000,"245,000.0000","5,568.1800",12,154,8.0000,0.6667,59.4200,12.4090,41,0,0,275.2900,1,182.3600,433.4460,4,120.0910,35,386.2050,3,205.0910,6,145.9130,53,"3,142.4800",41.1421,21,1,2017,2
2,2 ROOM,Improved,44.0000,"238,000.0000","5,409.0900",9,160,8.0000,0.8889,62.3300,12.3800,38,0,0,463.8570,1,128.5170,290.3890,5,197.6750,34,317.0190,1,313.6760,5,175.3860,54,"3,178.2000",154.1170,14,1,2017,2
3,3 ROOM,New Generation,74.0000,"333,000.0000","4,500.0000",11,138,5.0000,0.4545,60.1700,12.7159,40,0,0,"1,409.1200",0,"1,000.4700",429.0410,5,139.6380,11,709.2930,1,408.3930,2,155.6730,46,"3,427.4400",31.1346,17,1,2017,2
4,3 ROOM,New Generation,67.0000,"295,000.0000","4,402.9800",12,122,8.0000,0.6667,60.5000,12.5947,39,0,0,"1,345.6200",0,896.3680,420.9420,5,251.3010,12,630.9160,1,411.1180,2,120.1490,48,"3,388.6900",64.4931,22,1,2017,2


In [42]:
print("resale_df describe()")
resale_df.describe(include='all')

resale_df describe()


,flat_type,flat_model,floor_area_sqm,resale_price,price_per_sqm,max_floor_lvl,total_dwelling_units,storey_mid,storey_ratio,remaining_lease_years,log_resale_price,town,has_market_hawker,has_multistorey_carpark,dist_to_nearest_mrt_m,n_mrt_within_1km,dist_to_mall_m,dist_to_school_m,n_school_within_1km,dist_to_food_m,n_food_within_1km,dist_to_park_m,n_park_within_1km,dist_to_supermarket_m,n_supermarket_within_1km,dist_to_nearest_bus_stop_m,n_bus_stop_within_1km,dist_to_nearest_tourist_attraction_m,dist_to_nearest_carpark_m,n_carparks_within_500m,month_index,year,month
count,222392,222392,"222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000","222,392.0000"
unique,7,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,4 ROOM,Model A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,94130,79473,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,96.8700,"529,023.2022","5,521.7371",16.0395,127.5788,8.7746,0.5562,74.0816,13.1182,26.4199,0.0001,0.0002,620.9549,2.3275,536.1856,393.0552,4.2233,240.5017,23.3850,"1,202.8835",0.4937,358.3435,4.5074,115.5394,50.1305,"3,026.7973",86.1988,13.0694,57.9397,"2,021.3682",6.5211
std,NaN,NaN,24.0532,"188,740.0964","1,636.5659",6.8997,58.7500,5.9588,0.2817,14.1999,0.3480,14.2701,0.0095,0.0144,388.9034,2.3420,281.6064,258.1291,1.9929,139.0364,13.7945,576.8335,0.7463,202.5640,1.9154,60.3817,11.6728,"1,606.4286",51.7459,4.6181,30.7991,2.5895,3.4199
min,NaN,NaN,31.0000,"140,000.0000","2,089.5500",2.0000,2.0000,2.0000,0.0417,39.7500,11.8494,2.0000,0.0000,0.0000,5.5598,0.0000,1.1119,0.0000,0.0000,0.0000,0.0000,11.1195,0.0000,0.0000,0.0000,0.0000,7.0000,32.2465,0.0000,0.0000,1.0000,"2,017.0000",1.0000
25%,NaN,NaN,82.0000,"390,000.0000","4,363.6400",12.0000,92.0000,5.0000,0.3333,62.3300,12.8739,12.0000,0.0000,0.0000,333.7630,1.0000,333.5850,225.1560,3.0000,131.3960,14.0000,775.1450,0.0000,222.9480,3.0000,77.8364,42.0000,"1,747.1200",44.4780,10.0000,32.0000,"2,019.0000",4.0000
50%,NaN,NaN,93.0000,"500,000.0000","5,243.2000",14.0000,114.0000,8.0000,0.5333,73.8300,13.1224,27.0000,0.0000,0.0000,554.1430,1.0000,484.3950,340.5580,4.0000,224.1980,21.0000,"1,178.9100",0.0000,334.7890,4.0000,114.0680,49.0000,"2,853.2600",94.5157,13.0000,59.0000,"2,021.0000",7.0000
75%,NaN,NaN,112.0000,"635,000.0000","6,271.1900",17.0000,148.0000,11.0000,0.8000,88.2500,13.3614,38.0000,0.0000,0.0000,822.6110,3.0000,714.8530,482.5860,5.0000,326.4110,30.0000,"1,598.4500",1.0000,458.6130,6.0000,142.3300,58.0000,"4,225.1400",116.7200,16.0000,84.0000,"2,024.0000",9.0000


In [43]:
# Check resale_df datatypes
print("resale_df info()")
resale_df.info()

resale_df info()
<class 'pandas.DataFrame'>
RangeIndex: 222392 entries, 0 to 222391
Data columns (total 33 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   flat_type                             222392 non-null  str    
 1   flat_model                            222392 non-null  str    
 2   floor_area_sqm                        222392 non-null  float64
 3   resale_price                          222392 non-null  float64
 4   price_per_sqm                         222392 non-null  float64
 5   max_floor_lvl                         222392 non-null  int64  
 6   total_dwelling_units                  222392 non-null  int64  
 7   storey_mid                            222392 non-null  float64
 8   storey_ratio                          222392 non-null  float64
 9   remaining_lease_years                 222392 non-null  float64
 10  log_resale_price                      222392 non-null  float64

In [44]:
# Check for missing values
resale_df.isnull().sum()

flat_type                               0
flat_model                              0
floor_area_sqm                          0
resale_price                            0
price_per_sqm                           0
max_floor_lvl                           0
total_dwelling_units                    0
storey_mid                              0
storey_ratio                            0
remaining_lease_years                   0
log_resale_price                        0
town                                    0
has_market_hawker                       0
has_multistorey_carpark                 0
dist_to_nearest_mrt_m                   0
n_mrt_within_1km                        0
dist_to_mall_m                          0
dist_to_school_m                        0
n_school_within_1km                     0
dist_to_food_m                          0
n_food_within_1km                       0
dist_to_park_m                          0
n_park_within_1km                       0
dist_to_supermarket_m             

## Data Preparation

In [ ]:
features_selected = ["flat_model", "floor_area_sqm", "max_floor_lvl", "total_dwelling_units",
                     "storey_mid", "remaining_lease_years", "town",
                     "dist_to_nearest_mrt_m", "n_mrt_within_1km", "dist_to_nearest_bus_stop_m", 
                     "n_bus_stop_within_1km", "month_index", "dist_to_food_m", "n_food_within_1km",
                     "dist_to_supermarket_m", "n_supermarket_within_1km"]
# "dist_to_nearest_carpark_m"

categorical_columns = ["flat_model", "town"]

numerical_columns = [col for col in features_selected if col not in categorical_columns]

def regression_metrics(y_true, y_pred, label=""):
    """Return RMSE, MAE, MAPE, R² as a dict."""
    rmse  = root_mean_squared_error(y_true, y_pred)
    mae   = mean_absolute_error(y_true, y_pred)
    mape  = np.mean(np.abs((y_true - y_pred) / y_true)) * 100   # in %
    r2    = r2_score(y_true, y_pred)
    if label:
        print(f"\n{'─'*50}")
        print(f"  {label}")
        print(f"{'─'*50}")
        print(f"  RMSE : {rmse:>14,.2f}")
        print(f"  MAE  : {mae:>14,.2f}")
        print(f"  MAPE : {mape:>13.2f} %")
        print(f"  R²   : {r2:>14.4f}")
    return dict(rmse=rmse, mae=mae, mape=mape, r2=r2)

def make_preprocessor():
    # Numerical pipeline (imputer in case of missing values)
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaling", StandardScaler())
    ])

    # Categorical pipeline (imputer in case of missing values)
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer([
        ("num", num_pipeline, numerical_columns),
        ("cat", cat_pipeline, categorical_columns)
    ])

    return preprocessor

In [46]:
df = resale_df[features_selected + ["resale_price", "log_resale_price"]].dropna().copy()

X = df[features_selected]
y_raw = df["resale_price"]
y_log = df["log_resale_price"]

X_train, X_test, yr_train, yr_test, yl_train, yl_test = train_test_split(
    X, y_raw, y_log, test_size=0.2, random_state=42
)

In [47]:
result_dictionaries = []
models_used = []

## Random Forest Regression

In [48]:
# Random Forest Regression: Raw Price
rf_model_raw = Pipeline([
    ("preprocessing", make_preprocessor()),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model_raw.fit(X_train, yr_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

In [49]:
# Evaluate and print metrics for Random Forest Regression: Raw Price
rf_y_pred_raw = rf_model_raw.predict(X_test)
rf_raw_results = regression_metrics(yr_test, rf_y_pred_raw, "Random Forest  |  target = resale_price")

result_dictionaries.append(rf_raw_results)
models_used.append("Random Forest  |  target = resale_price")

# rf_raw_results


──────────────────────────────────────────────────
  Random Forest  |  target = resale_price
──────────────────────────────────────────────────
  RMSE :      61,026.96
  MAE  :      42,172.37
  MAPE :          8.00 %
  R²   :         0.8960


In [50]:
# Random Forest Regression: Log Price
rf_model_log = Pipeline([
    ("preprocessing", make_preprocessor()),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model_log.fit(X_train, yl_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

In [51]:
# Evaluate and print metrics 
rf_y_pred_log_raw = np.exp(rf_model_log.predict(X_test))

# Metrics in log-space (model quality)
rf_log_results = regression_metrics(
    yl_test, np.log(rf_y_pred_log_raw),
    "Random Forest  |  target = log_resale_price  [log-space metrics]"
)

# Metrics in dollar-amount (interpretable RMSE/MAE)
rf_log_raw_results = regression_metrics(
    yr_test, rf_y_pred_log_raw,
    "Random Forest  |  target = log_resale_price  [converted back to dollar amount]"
)

result_dictionaries.append(rf_log_results)
models_used.append("Random Forest  |  target = log_resale_price  [log-space metrics]")
result_dictionaries.append(rf_log_raw_results)
models_used.append("Random Forest  |  target = log_resale_price  [converted back to dollar amount]")

# print(rf_log_results)
# print(rf_log_raw_results)


──────────────────────────────────────────────────
  Random Forest  |  target = log_resale_price  [log-space metrics]
──────────────────────────────────────────────────
  RMSE :           0.11
  MAE  :           0.08
  MAPE :          0.60 %
  R²   :         0.9063

──────────────────────────────────────────────────
  Random Forest  |  target = log_resale_price  [converted back to dollar amount]
──────────────────────────────────────────────────
  RMSE :      61,411.05
  MAE  :      41,569.25
  MAPE :          7.77 %
  R²   :         0.8947


## XGBoost Regression

In [52]:
# XGBoost Regression: Raw Price
xgb_model_raw = Pipeline([
    ("preprocessing", make_preprocessor()),
    ("model", XGBRegressor(
        objective='reg:squarederror',
        n_estimators=100,
        random_state=42,
        learning_rate=0.1
    ))
])

xgb_model_raw.fit(X_train, yr_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

In [53]:
# Evaluate and print metrics for XGBoost Regression: Raw Price
xgb_y_pred_raw = xgb_model_raw.predict(X_test)
xgb_raw_results = regression_metrics(yr_test, xgb_y_pred_raw, "XGBoost  |  target = resale_price")

result_dictionaries.append(xgb_raw_results)
models_used.append("XGBoost  |  target = resale_price")

# xgb_raw_results


──────────────────────────────────────────────────
  XGBoost  |  target = resale_price
──────────────────────────────────────────────────
  RMSE :      53,676.87
  MAE  :      37,463.44
  MAPE :          7.10 %
  R²   :         0.9195


In [54]:
# XGBoost Regression: Log Price
xgb_model_log = Pipeline([
    ("preprocessing", make_preprocessor()),
    ("model", XGBRegressor(
        objective='reg:squarederror',
        n_estimators=100,
        random_state=42,
        learning_rate=0.1
    ))
])

xgb_model_log.fit(X_train, yl_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

In [55]:
# Evaluate and print metrics 
xgb_y_pred_log_raw = np.exp(xgb_model_log.predict(X_test))

# Metrics in log-space (model quality)
xgb_log_results = regression_metrics(
    yl_test, np.log(xgb_y_pred_log_raw),
    "XGBoost  |  target = log_resale_price  [log-space metrics]"
)

# Metrics in dollar-amount (interpretable RMSE/MAE, MAPE)
xgb_log_raw_results = regression_metrics(
    yr_test, xgb_y_pred_log_raw,
    "XGBoost  |  target = log_resale_price  [converted back to dollar amount]"
)

result_dictionaries.append(xgb_log_results)
models_used.append("XGBoost  |  target = log_resale_price  [log-space metrics]")
result_dictionaries.append(xgb_log_raw_results)
models_used.append("XGBoost  |  target = log_resale_price  [converted back to dollar amount]")

# print(xgb_log_results)
# print(xgb_log_raw_results)


──────────────────────────────────────────────────
  XGBoost  |  target = log_resale_price  [log-space metrics]
──────────────────────────────────────────────────
  RMSE :           0.09
  MAE  :           0.07
  MAPE :          0.53 %
  R²   :         0.9274

──────────────────────────────────────────────────
  XGBoost  |  target = log_resale_price  [converted back to dollar amount]
──────────────────────────────────────────────────
  RMSE :      54,466.77
  MAE  :      37,124.26
  MAPE :          6.90 %
  R²   :         0.9171


## Compare results

In [57]:
result_df = pd.DataFrame([
    {"model": model, **metrics}
    for model, metrics in zip(models_used, result_dictionaries)
])

result_df

,model,rmse,mae,mape,r2
0,Random Forest | target = resale_price,"61,026.9564","42,172.3695",7.9967,0.8960
1,Random Forest | target = log_resale_price [...,0.1069,0.0783,0.5962,0.9063
2,Random Forest | target = log_resale_price [...,"61,411.0453","41,569.2496",7.7698,0.8947
3,XGBoost | target = resale_price,"53,676.8745","37,463.4421",7.0985,0.9195
4,XGBoost | target = log_resale_price [log-sp...,0.0941,0.0694,0.5282,0.9274
5,XGBoost | target = log_resale_price [conver...,"54,466.7659","37,124.2564",6.8992,0.9171


In [58]:
HDB_db.close()